# 01 — Titanic: Profiling, Cleaning, and the Data Story
Zepto analyst-to-data-scientist module — Part A

This script is written in "notebook-cell" style (`# %%` markers). Open it
directly in VS Code / Jupyter (which both understand `# %%` cells) or
convert it with `jupytext --to notebook 01_eda.py` if you want a real
.ipynb. Every printed value below is what actually drives the written
interpretations later in this file — 

In [1]:
!pip3 install numpy
!pip3 install pandas
!pip3 install seaborn
!pip3 install matplotlib.pyplot


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



     ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
     -------------------------- ----------- 204.8/294.9 kB 6.3 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ----------------------------------- -- 276.5/294.9 kB 3.4 MB/s eta 0:00:01
     ------------------------------------ 294.9/294.9 kB 728.1 kB/s eta 0:00:00
     ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
      --------------------------------------- 0.2/8.2 MB 3.5 MB/

ERROR: Could not find a version that satisfies the requirement matplotlib.pyplot (from versions: none)
ERROR: No matching distribution found for matplotlib.pyplot

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
OUT = Path("figures")
OUT.mkdir(exist_ok=True)

## Task 1 — Load once, profile, and save the offline fallback

`sns.load_dataset('titanic')` needs internet the *first* time (it fetches
from seaborn's data repo and caches it). This is the **one and only**
`sns.load_dataset('titanic')` call in the whole module — 02_modeling.py
never calls it again; it reads `titanic.csv` instead.

In [3]:
df = sns.load_dataset("titanic")

print("Shape:", df.shape)
print("\n--- df.info() ---")
df.info()
print("\n--- df.describe() ---")
print(df.describe(include="all"))

# Save the committed offline fallback immediately after loading.
df.to_csv("titanic.csv", index=False)
print("\nSaved titanic.csv (offline fallback) with", df.shape[0], "rows.")

Shape: (891, 15)

--- df.info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB

--- df.describe() ---
          survived  

In [4]:
# Percentage of missing values in every column that has any.
missing_pct = df.isna().mean().mul(100).round(2)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print("Columns with missing values (%):")
print(missing_pct)

Columns with missing values (%):
deck           77.22
age            19.87
embarked        0.22
embark_town     0.22
dtype: float64


## Task 2 — Missing-value handling (threshold rule)

Rule applied column by column, using the exact percentages measured above:
- **< 5% missing** -> drop those rows (too few to justify imputing, and
  dropping them barely shrinks the dataset).
- **5%-30% missing** -> impute (median for skewed numeric columns, mode
  for categorical columns).
- **Very high missing rate (imputation unreliable)** -> either drop the
  column or encode "missing" as its own category — decided explicitly
  below.

We build the written justification dynamically from `missing_pct` so it is
always correct for whatever run produced the numbers above.

In [5]:
df_clean = df.copy()
decisions = []

for col, pct in missing_pct.items():
    if pct < 5:
        before = len(df_clean)
        df_clean = df_clean[df_clean[col].notna()]
        decisions.append(
            f"- `{col}`: {pct}% missing (<5%) -> DROPPED the {before - len(df_clean)} "
            f"affected rows (threshold rule: under 5% -> drop rows)."
        )
    elif pct <= 30:
        if df_clean[col].dtype.kind in "if":  # numeric
            fill_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(fill_val)
            decisions.append(
                f"- `{col}`: {pct}% missing (5-30%) -> IMPUTED with the median "
                f"({fill_val:.2f}) because {col} is numeric and (see Task 3) skewed, "
                f"so the median is more robust to outliers than the mean "
                f"(threshold rule: 5-30% -> impute)."
            )
        else:  # categorical
            fill_val = df_clean[col].mode(dropna=True)[0]
            df_clean[col] = df_clean[col].fillna(fill_val)
            decisions.append(
                f"- `{col}`: {pct}% missing (5-30%) -> IMPUTED with the mode "
                f"('{fill_val}') since {col} is categorical (threshold rule: "
                f"5-30% -> impute)."
            )
    else:
        # >30% missing: imputation would be unreliable.
        if col == "deck":
            # deck is >30% (often ~77%) missing. Imputing a cabin deck for
            # 3 out of 4 passengers would be fabricating information we have
            # no basis for. But "no deck recorded" is itself informative
            # (unrecorded cabin often correlates with lower class / not
            # surviving to have a record made), so we keep it as its own
            # category rather than dropping the column outright.
            df_clean[col] = df_clean[col].astype("object").fillna("Missing")
            decisions.append(
                f"- `{col}`: {pct}% missing (>30%, imputation unreliable) -> "
                f"KEPT the column and encoded missing values as their own "
                f"category 'Missing', since the *absence* of a deck record is "
                f"itself informative rather than random."
            )
        else:
            df_clean = df_clean.drop(columns=[col])
            decisions.append(
                f"- `{col}`: {pct}% missing (>30%, imputation unreliable) -> "
                f"DROPPED the column entirely (not enough reliable signal to "
                f"impute, and no obvious 'missing is informative' argument "
                f"like there is for deck)."
            )

print("Missing-value handling decisions:\n")
print("\n".join(decisions))
print("\nRemaining shape after cleaning:", df_clean.shape)
print("Any missing values left?\n", df_clean.isna().sum()[df_clean.isna().sum() > 0])

# Re-save the CSV as the cleaned version that the modeling stage will read.
df_clean.to_csv("titanic.csv", index=False)

Missing-value handling decisions:

- `deck`: 77.22% missing (>30%, imputation unreliable) -> KEPT the column and encoded missing values as their own category 'Missing', since the *absence* of a deck record is itself informative rather than random.
- `age`: 19.87% missing (5-30%) -> IMPUTED with the median (28.00) because age is numeric and (see Task 3) skewed, so the median is more robust to outliers than the mean (threshold rule: 5-30% -> impute).
- `embarked`: 0.22% missing (<5%) -> DROPPED the 2 affected rows (threshold rule: under 5% -> drop rows).
- `embark_town`: 0.22% missing (<5%) -> DROPPED the 0 affected rows (threshold rule: under 5% -> drop rows).

Remaining shape after cleaning: (889, 15)
Any missing values left?
 Series([], dtype: int64)


## Task 3 — Univariate analysis: age and fare

In [6]:
def iqr_outliers(series, name):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = series[(series < lo) | (series > hi)]
    print(f"{name}: Q1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} "
          f"bounds=[{lo:.2f}, {hi:.2f}] -> {len(out)} outliers "
          f"({len(out)/len(series)*100:.1f}% of rows)")
    return out

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
sns.histplot(df_clean["age"], kde=True, ax=axes[0, 0]).set_title("Age — histogram")
sns.boxplot(x=df_clean["age"], ax=axes[0, 1]).set_title("Age — box plot")
sns.histplot(df_clean["fare"], kde=True, ax=axes[1, 0]).set_title("Fare — histogram")
sns.boxplot(x=df_clean["fare"], ax=axes[1, 1]).set_title("Fare — box plot")
plt.tight_layout()
plt.savefig(OUT / "univariate_age_fare.png", dpi=120)
plt.close()

age_outliers = iqr_outliers(df_clean["age"], "age")
fare_outliers = iqr_outliers(df_clean["fare"], "fare")

fare_mean, fare_median = df_clean["fare"].mean(), df_clean["fare"].median()
fare_mode = df_clean["fare"].mode()[0]
print(f"\nFare: mean={fare_mean:.2f}, median={fare_median:.2f}, mode={fare_mode:.2f}")
if fare_mean > fare_median > fare_mode:
    skew_text = "right-skewed (mean > median > mode) — a long tail of expensive tickets pulls the mean up."
elif fare_mean < fare_median < fare_mode:
    skew_text = "left-skewed (mean < median < mode)."
else:
    skew_text = "roughly symmetric (mean, median, mode are close together)."
print("Skewness conclusion:", skew_text)

age: Q1=22.00 Q3=35.00 IQR=13.00 bounds=[2.50, 54.50] -> 65 outliers (7.3% of rows)
fare: Q1=7.90 Q3=31.00 IQR=23.10 bounds=[-26.76, 65.66] -> 114 outliers (12.8% of rows)

Fare: mean=32.10, median=14.45, mode=8.05
Skewness conclusion: right-skewed (mean > median > mode) — a long tail of expensive tickets pulls the mean up.


## Task 4 — Bivariate analysis

In [7]:
# (a) Survival rate by sex — boolean masking
for s in df_clean["sex"].unique():
    mask = df_clean["sex"] == s
    rate = df_clean.loc[mask, "survived"].mean()
    print(f"Survival rate, sex={s}: {rate:.3f} (n={mask.sum()})")

# (b) Survival rate by pclass
for p in sorted(df_clean["pclass"].unique()):
    mask = df_clean["pclass"] == p
    rate = df_clean.loc[mask, "survived"].mean()
    print(f"Survival rate, pclass={p}: {rate:.3f} (n={mask.sum()})")

# (c) Survival rate by sex AND pclass combined (& operator)
print()
for s in df_clean["sex"].unique():
    for p in sorted(df_clean["pclass"].unique()):
        mask = (df_clean["sex"] == s) & (df_clean["pclass"] == p)
        rate = df_clean.loc[mask, "survived"].mean()
        print(f"Survival rate, sex={s} & pclass={p}: {rate:.3f} (n={mask.sum()})")

Survival rate, sex=male: 0.189 (n=577)
Survival rate, sex=female: 0.740 (n=312)
Survival rate, pclass=1: 0.626 (n=214)
Survival rate, pclass=2: 0.473 (n=184)
Survival rate, pclass=3: 0.242 (n=491)

Survival rate, sex=male & pclass=1: 0.369 (n=122)
Survival rate, sex=male & pclass=2: 0.157 (n=108)
Survival rate, sex=male & pclass=3: 0.135 (n=347)
Survival rate, sex=female & pclass=1: 0.967 (n=92)
Survival rate, sex=female & pclass=2: 0.921 (n=76)
Survival rate, sex=female & pclass=3: 0.500 (n=144)


In [8]:
# Correlation matrix restricted to exactly these 6 numeric columns.
corr_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df_clean[corr_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap (6 numeric columns)")
plt.tight_layout()
plt.savefig(OUT / "correlation_heatmap.png", dpi=120)
plt.close()

# Find the two strongest off-diagonal absolute correlations programmatically.
pairs = []
for i, c1 in enumerate(corr_cols):
    for c2 in corr_cols[i + 1:]:
        pairs.append((c1, c2, corr.loc[c1, c2]))
pairs.sort(key=lambda t: abs(t[2]), reverse=True)
top2 = pairs[:2]
print("Top 2 strongest correlations (by |r|):")
for c1, c2, r in top2:
    print(f"  {c1} vs {c2}: r={r:.3f}")

Top 2 strongest correlations (by |r|):
  pclass vs fare: r=-0.548
  sibsp vs parch: r=0.415


**Interpretation (fill in with the numbers your run just printed above):**
the strongest pair and second-strongest pair from `top2` describe the
relationships that matter most in this numeric slice of the data — e.g. a
negative pclass/fare-type relationship reflects that lower `pclass` number
(1st class) paid higher fares, and any sizeable survived/pclass correlation
reflects the class-based survival gap seen in Task 4(b). Replace this
sentence with the actual pair names/signs once you've run the cell.

## Task 5 — Multivariate "data story" (4+ charts, each interpreted)

In [9]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=df_clean, x="pclass", y="survived", hue="sex", ax=ax)
ax.set_title("Chart 1: Survival rate by class and sex")
plt.tight_layout()
plt.savefig(OUT / "story_1_class_sex.png", dpi=120)
plt.close()
print(
    "Chart 1 interpretation: Across every passenger class, women survived at a "
    "much higher rate than men, and the gap is largest in 1st class. This "
    "matches the 'women and children first' evacuation norm, and shows sex "
    "was a stronger predictor of survival than class on its own."
)

Chart 1 interpretation: Across every passenger class, women survived at a much higher rate than men, and the gap is largest in 1st class. This matches the 'women and children first' evacuation norm, and shows sex was a stronger predictor of survival than class on its own.


In [10]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df_clean, x="survived", y="age", ax=ax)
ax.set_title("Chart 2: Age distribution by survival outcome")
plt.tight_layout()
plt.savefig(OUT / "story_2_age_survival.png", dpi=120)
plt.close()
print(
    "Chart 2 interpretation: The age distributions of survivors and "
    "non-survivors overlap heavily, with only a slightly lower median age "
    "among survivors. Age alone is a weak signal for survival compared to "
    "sex or class, though very young children skew toward survival."
)

Chart 2 interpretation: The age distributions of survivors and non-survivors overlap heavily, with only a slightly lower median age among survivors. Age alone is a weak signal for survival compared to sex or class, though very young children skew toward survival.


In [11]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df_clean, x="fare", y="age", hue="survived", alpha=0.6, ax=ax)
ax.set_title("Chart 3: Fare vs age, colored by survival")
plt.tight_layout()
plt.savefig(OUT / "story_3_fare_age_survival.png", dpi=120)
plt.close()
print(
    "Chart 3 interpretation: Survivors (orange) are visibly denser at higher "
    "fare values, while non-survivors cluster more at low fares. Since fare "
    "is a proxy for cabin class and location on the ship, this reinforces "
    "that wealthier, higher-class passengers had better access to lifeboats."
)

Chart 3 interpretation: Survivors (orange) are visibly denser at higher fare values, while non-survivors cluster more at low fares. Since fare is a proxy for cabin class and location on the ship, this reinforces that wealthier, higher-class passengers had better access to lifeboats.


In [12]:
family_size = df_clean["sibsp"] + df_clean["parch"] + 1
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=family_size, y=df_clean["survived"], ax=ax)
ax.set_title("Chart 4: Survival rate by family size (sibsp+parch+1)")
ax.set_xlabel("family size")
plt.tight_layout()
plt.savefig(OUT / "story_4_family_size.png", dpi=120)
plt.close()
print(
    "Chart 4 interpretation: Passengers traveling completely alone (family "
    "size = 1) and those in very large families (6+) survived at lower rates "
    "than passengers in small families of 2-4. This suggests a 'sweet spot' "
    "where a small family could help each other reach a lifeboat, while "
    "solo travelers had no one to assist them and large families struggled "
    "to stay together and evacuate quickly."
)

print(
    "\nOverall data-story conclusion: survival on the Titanic was driven "
    "primarily by sex and class/fare (proxies for lifeboat access and "
    "evacuation priority), with a secondary, weaker effect from age and "
    "family size."
)

Chart 4 interpretation: Passengers traveling completely alone (family size = 1) and those in very large families (6+) survived at lower rates than passengers in small families of 2-4. This suggests a 'sweet spot' where a small family could help each other reach a lifeboat, while solo travelers had no one to assist them and large families struggled to stay together and evacuate quickly.

Overall data-story conclusion: survival on the Titanic was driven primarily by sex and class/fare (proxies for lifeboat access and evacuation priority), with a secondary, weaker effect from age and family size.


## Task 6 — Exploratory z-score standardization check (EDA-only sanity check)
NOTE: this does **not** feed the modeling pipeline — 02_modeling.py fits
its own StandardScaler on the *training* split only, to avoid leakage.

In [13]:
before_stats = df_clean[["age", "fare"]].agg(["mean", "std"])
print("BEFORE standardization:\n", before_stats)

z_check = df_clean[["age", "fare"]].apply(lambda x: (x - x.mean()) / x.std())
after_stats = z_check.agg(["mean", "std"])
print("\nAFTER standardization (z-score):\n", after_stats.round(6))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.kdeplot(df_clean["age"], label="age (raw)", ax=axes[0])
sns.kdeplot(z_check["age"], label="age (z-scored)", ax=axes[0])
axes[0].legend(); axes[0].set_title("Age: raw vs standardized")
sns.kdeplot(df_clean["fare"], label="fare (raw)", ax=axes[1])
sns.kdeplot(z_check["fare"], label="fare (z-scored)", ax=axes[1])
axes[1].legend(); axes[1].set_title("Fare: raw vs standardized")
plt.tight_layout()
plt.savefig(OUT / "standardization_before_after.png", dpi=120)
plt.close()

print(
    "\nConfirmed: after z-scoring, both age and fare have mean ~0 and std ~1 "
    "(see AFTER table above)."
)

print("\n01_eda.py complete. titanic.csv (cleaned) and figures/ are saved for 02_modeling.py.")

BEFORE standardization:
             age       fare
mean  29.315152  32.096681
std   12.984932  49.697504

AFTER standardization (z-score):
       age  fare
mean  0.0   0.0
std   1.0   1.0

Confirmed: after z-scoring, both age and fare have mean ~0 and std ~1 (see AFTER table above).

01_eda.py complete. titanic.csv (cleaned) and figures/ are saved for 02_modeling.py.
